<a href="https://colab.research.google.com/github/MR-just01/Llama3.2-Reasoning/blob/main/notebooks/04(qwen).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/MR-just01/Llama3.2-Reasoning

fatal: destination path 'Llama3.2-Reasoning' already exists and is not an empty directory.


In [2]:
%cd Llama3.2-Reasoning
!ls

/content/Llama3.2-Reasoning
data  LICENSE  notebooks  outputs  README.md  reports  requirements.txt  src


In [3]:
!find data

data
data/splits
data/splits/ReadME.md
data/raw
data/raw/ReadME.md
data/processed
data/processed/startegyQA_standardized.csv
data/processed/arc_challenge_standardized.csv
data/processed/reasoning_dataset (1).csv
data/processed/aqua_rat_standardized.csv
data/processed/gsm8k_standardized.csv


In [4]:
import pandas as pd

df = pd.read_csv("data/processed/reasoning_dataset (1).csv",
                   keep_default_na=False)

print(df.shape)
df.head()

(30193, 6)


,instruction,input,reasoning,answer,dataset,task_type
0,Solve the following math reasoning problem ste...,Nicole collected 400 Pokemon cards. Cindy coll...,Cindy has 400 x 2 = <<400*2=800>>800 cards.\nN...,150,gsm8k,math_reasoning
1,Solve the following math reasoning problem ste...,James had two browsers on his computer. In eac...,The total number of tabs in three windows of e...,60,gsm8k,math_reasoning
2,Solve the following multiple-choice math reaso...,The 100-milliliter solution of sugar and water...,In the original solution the amount of sugar i...,50,AQUA-RAT,math_reasoning
3,Solve the following multiple-choice math reaso...,"If 0.75 : x :: 5 : 8, then x is equal to:\n\nC...",Explanation:\n(x x 5) = (0.75 x 8)\nx=6/5\n=1....,1.2,AQUA-RAT,math_reasoning
4,Solve the following multiple-choice math reaso...,The price of Darjeeling tea (in rupees per kil...,Explanation :\nPrice of Darjeeling tea (in rup...,May 20,AQUA-RAT,math_reasoning


In [5]:
print(df.info())
print(df.isnull().sum())

print(df["dataset"].value_counts())

print(df["task_type"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30193 entries, 0 to 30192
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  30193 non-null  object
 1   input        30193 non-null  object
 2   reasoning    30193 non-null  object
 3   answer       30193 non-null  object
 4   dataset      30193 non-null  object
 5   task_type    30193 non-null  object
dtypes: object(6)
memory usage: 1.4+ MB
None
instruction    0
input          0
reasoning      0
answer         0
dataset        0
task_type      0
dtype: int64
dataset
AQUA-RAT         19999
gsm8k             7473
StrategyQA        1603
ARC-Challenge     1118
Name: count, dtype: int64
task_type
math_reasoning           27472
commonsense_reasoning     1603
science_reasoning         1118
Name: count, dtype: int64


In [6]:
print(df.duplicated().sum())
sample = df.iloc[0]

print(sample["instruction"])
print()

print(sample["input"])
print()

print(sample["reasoning"])
print()

# print(sample["answer"])
print(df.loc[0, "dataset"])
print()

print(df.loc[0, "reasoning"])

0
Solve the following math reasoning problem step by step.

Nicole collected 400 Pokemon cards. Cindy collected twice as many, and Rex collected half of Nicole and Cindy's combined total. If Rex divided his card equally among himself and his three younger siblings, how many cards does Rex have left?

Cindy has 400 x 2 = <<400*2=800>>800 cards.
Nicole and Cindy have 400 + 800 = <<400+800=1200>>1200 cards.
Rex has 1200/2 = <<1200/2=600>>600 cards.
Rex is left with 600/(3+1=4) = <<600/4=150>>150 cards

gsm8k

Cindy has 400 x 2 = <<400*2=800>>800 cards.
Nicole and Cindy have 400 + 800 = <<400+800=1200>>1200 cards.
Rex has 1200/2 = <<1200/2=600>>600 cards.
Rex is left with 600/(3+1=4) = <<600/4=150>>150 cards


In [10]:
!pip install -q --upgrade pip

!pip install -q \
transformers \
datasets \
accelerate \
bitsandbytes \
trl \
peft \
sentencepiece \
pyarrow==17.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.0 MB/s eta 0:00:00


In [7]:
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
)

from trl import SFTTrainer

In [8]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [9]:
def format_prompt(row):

    messages = [
        {
            "role": "user",
            "content":
                f"{row['instruction']}\n\n{row['input']}"
        },
        {
            "role": "assistant",
            "content":
                f"Reasoning:\n{row['reasoning']}\n\n"
                f"Final Answer:\n{row['answer']}"
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

In [10]:
df["text"] = df.apply(format_prompt, axis=1)

In [11]:
print(df["text"].iloc[0])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Solve the following math reasoning problem step by step.

Nicole collected 400 Pokemon cards. Cindy collected twice as many, and Rex collected half of Nicole and Cindy's combined total. If Rex divided his card equally among himself and his three younger siblings, how many cards does Rex have left?<|im_end|>
<|im_start|>assistant
Reasoning:
Cindy has 400 x 2 = <<400*2=800>>800 cards.
Nicole and Cindy have 400 + 800 = <<400+800=1200>>1200 cards.
Rex has 1200/2 = <<1200/2=600>>600 cards.
Rex is left with 600/(3+1=4) = <<600/4=150>>150 cards

Final Answer:
150<|im_end|>



In [39]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [40]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

In [41]:
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
print("Model prepared for k-bit training.")

Model prepared for k-bit training.


In [42]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [43]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)

In [44]:
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [45]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="outputs",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio= 0.03,
    report_to="none",

    # These moved here
    dataset_text_field="text",
    max_length=1024,
    packing=False,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


# Split the dataset

In [46]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)
print("Training:", train_df.shape)
print("Validation:", val_df.shape)

Training: (27173, 7)
Validation: (3020, 7)


# Convert to the hugging face dataset

In [47]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['instruction', 'input', 'reasoning', 'answer', 'dataset', 'task_type', 'text'],
    num_rows: 27173
})
Dataset({
    features: ['instruction', 'input', 'reasoning', 'answer', 'dataset', 'task_type', 'text'],
    num_rows: 3020
})


In [48]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/27173 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/27173 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/27173 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/3020 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/3020 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/3020 [00:00<?, ? examples/s]

In [50]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

2.11.0+cu128
12.8
Tesla T4


In [51]:
print(model.dtype)

torch.float32


In [52]:
print(training_args.fp16)
print(training_args.bf16)

True
False


In [53]:
import transformers
import trl
import bitsandbytes
import accelerate
import peft

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

Transformers: 5.14.1
TRL: 0.29.1
PEFT: 0.20.0
Accelerate: 1.14.0
BitsAndBytes: 0.50.0
